In [ ]:
# Cell 1: mount Drive and install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q huggingface_hub pandas

In [ ]:
# Cell 2: download RouterBench's released data
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id="withmartian/routerbench", repo_type="dataset")
import os
print(path, os.listdir(path))

In [ ]:
# Cell 3: load the 0-shot file and look at it
import pandas as pd, glob
f = glob.glob(path + "/*0shot*.pkl")[0]
df = pd.read_pickle(f)
print(df.shape)
print(list(df.columns))
print(df['eval_name'].value_counts())

In [ ]:
# Cell 4: keep only what we need
keep = ['sample_id', 'prompt', 'eval_name',
        'gpt-4-1106-preview', 'mistralai/mixtral-8x7b-chat',
        'gpt-4-1106-preview|total_cost', 'mistralai/mixtral-8x7b-chat|total_cost']
small = df[keep].copy()
small.columns = ['sample_id', 'prompt', 'dataset',
                 'score_strong', 'score_weak', 'cost_strong', 'cost_weak']

# Cell 5: sanity checks
print(small[['score_strong', 'score_weak', 'cost_strong', 'cost_weak']].describe())
print(small.isna().sum())
for name in ['grade-school-math', 'arc-challenge']:
    sub = small[small.dataset == name]
    print(name, len(sub),
          'strong acc', round(sub.score_strong.mean(), 3),
          'weak acc', round(sub.score_weak.mean(), 3),
          'strong cost', round(sub.cost_strong.sum(), 2),
          'weak cost', round(sub.cost_weak.sum(), 2))
    print(repr(sub.prompt.iloc[0][:600]))
    print('unique scores:', sorted(sub.score_strong.dropna().unique())[:10])

In [ ]:
# Cell 6: save the cleaned file inside MyDrive/MLResearch
import os
folder = '/content/drive/MyDrive/MLResearch'
os.makedirs(folder, exist_ok=True)

chosen = small[small.dataset.isin(['grade-school-math', 'arc-challenge'])].dropna().reset_index(drop=True)
chosen.to_csv(f'{folder}/routerbench_clean.csv', index=False)
print(chosen.shape)
print(chosen.dataset.value_counts())
print(os.listdir(folder))

In [ ]:
# Cell 7: first look at the end of the GSM8K prompts (check before cleaning)
g = chosen[chosen.dataset == 'grade-school-math']
for i in [0, 1, 2]:
    print(repr(g.prompt.iloc[i][-350:]))
    print('---')

In [ ]:
# Cell 8: extract just the question
import re

def extract_question(prompt, dataset):
    p = prompt.strip()
    if p.startswith("['") or p.startswith('["'):
        p = p[2:]
    if p.endswith("']") or p.endswith('"]'):
        p = p[:-2]
    p = p.replace('\\n', '\n')
    if dataset == 'grade-school-math':
        q = p.split('Question:')[-1]
        q = re.split(r'\nAnswer:', q)[0]
        return q.strip()
    else:  # arc-challenge
        q = p.split('Print only a single choice')[0]
        return q.strip()

chosen['question'] = [extract_question(p, d) for p, d in zip(chosen.prompt, chosen.dataset)]
for d in ['grade-school-math', 'arc-challenge']:
    print(d)
    for q in chosen[chosen.dataset == d].question.iloc[:3]:
        print(repr(q[:300]))
print(chosen.question.str.len().describe())

In [ ]:
# Cell 8 (replacement): parse the stored list and take the last element
import ast

def extract_question(prompt, dataset):
    try:
        parts = ast.literal_eval(prompt)
        p = parts[-1] if isinstance(parts, (list, tuple)) else str(parts)
    except Exception:
        p = prompt.strip().strip("[]").strip("'\"")
        p = p.replace('\\n', '\n')
    if dataset == 'arc-challenge':
        p = p.split('Print only a single choice')[0]
    return p.strip()

chosen['question'] = [extract_question(p, d) for p, d in zip(chosen.prompt, chosen.dataset)]

for d in ['grade-school-math', 'arc-challenge']:
    sub = chosen[chosen.dataset == d]
    print(d, 'unique questions:', sub.question.nunique(), 'of', len(sub))
    for q in sub.question.iloc[:3]:
        print(repr(q[:250]))
print(chosen.question.str.len().describe())

In [ ]:
# Cell 9a: install without the serve/eval extras
!pip install -q routellm

In [ ]:
import os, pandas as pd
from tqdm import tqdm

folder = '/content/drive/MyDrive/MLResearch'
part_path = f'{folder}/bert_scores_partial.csv'

# resume if a partial file exists
done = pd.read_csv(part_path) if os.path.exists(part_path) else pd.DataFrame(columns=['row', 'bert_score'])
scores = dict(zip(done.row, done.bert_score))
print('already scored:', len(scores))

todo = [i for i in range(len(chosen)) if i not in scores]
for n, i in enumerate(tqdm(todo)):
    scores[i] = bert.calculate_strong_win_rate(chosen.question.iloc[i])
    if (n + 1) % 500 == 0:
        pd.DataFrame({'row': list(scores.keys()), 'bert_score': list(scores.values())}).to_csv(part_path, index=False)

pd.DataFrame({'row': list(scores.keys()), 'bert_score': list(scores.values())}).to_csv(part_path, index=False)
chosen['bert_score'] = [scores[i] for i in range(len(chosen))]
chosen.to_csv(f'{folder}/routerbench_scored.csv', index=False)
print(chosen.groupby('dataset').bert_score.describe())

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
# Cell A: mount Drive, install, check the GPU
from google.colab import drive
drive.mount('/content/drive')
!pip install -q routellm huggingface_hub pandas
import torch
print('GPU available:', torch.cuda.is_available())

In [ ]:
# Cell B: load the cleaned file and rebuild the questions
import pandas as pd, ast

folder = '/content/drive/MyDrive/MLResearch'
chosen = pd.read_csv(f'{folder}/routerbench_clean.csv')

def extract_question(prompt, dataset):
    try:
        parts = ast.literal_eval(prompt)
        p = parts[-1] if isinstance(parts, (list, tuple)) else str(parts)
    except Exception:
        p = prompt.strip().strip("[]").strip("'\"").replace('\\n', '\n')
    if dataset == 'arc-challenge':
        p = p.split('Print only a single choice')[0]
    return p.strip()

chosen['question'] = [extract_question(p, d) for p, d in zip(chosen.prompt, chosen.dataset)]
print(chosen.shape, chosen.question.nunique())

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.is_available())

In [ ]:
import os, time
import pandas as pd
from tqdm import tqdm

folder = '/content/drive/MyDrive/MLResearch'

# fixed-seed random sample: 600 GSM8K + 400 ARC (change the numbers if you want more or less)
sample = pd.concat([
    chosen[chosen.dataset == 'grade-school-math'].sample(300, random_state=42),
    chosen[chosen.dataset == 'arc-challenge'].sample(200, random_state=42),
]).sort_index()

part_path = f'{folder}/bert_scores_sample_partial.csv'
done = pd.read_csv(part_path) if os.path.exists(part_path) else pd.DataFrame(columns=['row', 'bert_score'])
scores = dict(zip(done.row, done.bert_score))

todo = [i for i in sample.index if i not in scores]
for n, i in enumerate(tqdm(todo)):
    scores[i] = bert.calculate_strong_win_rate(chosen.question.loc[i])
    if (n + 1) % 100 == 0:
        pd.DataFrame({'row': list(scores.keys()), 'bert_score': list(scores.values())}).to_csv(part_path, index=False)

pd.DataFrame({'row': list(scores.keys()), 'bert_score': list(scores.values())}).to_csv(part_path, index=False)
sample['bert_score'] = [scores[i] for i in sample.index]
sample.to_csv(f'{folder}/routerbench_scored_sample.csv', index=False)
print(len(sample))
print(sample.groupby('dataset').bert_score.describe())

In [ ]:
import numpy as np, pandas as pd
folder = '/content/drive/MyDrive/MLResearch'
df = pd.read_csv(f'{folder}/routerbench_scored_sample.csv').reset_index(drop=True)

S, W = df.score_strong.values, df.score_weak.values
CS, CW = df.cost_strong.values, df.cost_weak.values
sc = df.bert_score.values
n = len(df)

def run(idx, share):
    """route the top `share` of prompts (by BERT score) to the strong model"""
    s, w, cs, cw, b = S[idx], W[idx], CS[idx], CW[idx], sc[idx]
    k = int(round(share * len(idx)))
    order = np.argsort(-b)
    to_strong = np.zeros(len(idx), bool); to_strong[order[:k]] = True
    acc = np.where(to_strong, s, w).mean()
    cost = np.where(to_strong, cs, cw).sum()
    return acc, cost, to_strong

def random_expect(idx, share):
    s, w, cs, cw = S[idx], W[idx], CS[idx], CW[idx]
    return share*s.mean() + (1-share)*w.mean(), share*cs.sum() + (1-share)*cw.sum()

def saved_at_matching_acc(idx, acc_r, cost_r):
    """cost of randomly mixing strong/weak to reach the router's accuracy, minus router cost"""
    a_s, a_w = S[idx].mean(), W[idx].mean()
    q = np.clip((acc_r - a_w) / (a_s - a_w), 0, 1)
    c_rand = q*CS[idx].sum() + (1-q)*CW[idx].sum()
    return c_rand - cost_r, c_rand

allidx = np.arange(n)
shares = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
rows = []
for sh in shares:
    acc, cost, ts = run(allidx, sh)
    racc, rcost = random_expect(allidx, sh)
    saved, c_rand = saved_at_matching_acc(allidx, acc, cost)
    thr = np.sort(sc)[::-1][int(round(sh*n))-1]          # cut-off score on OUR prompts
    rows.append(dict(target_share=sh, cutoff_score=round(thr,5),
        actual_share=ts.mean().round(3),
        share_gsm8k=ts[(df.dataset=='grade-school-math').values].mean().round(3),
        share_arc=ts[(df.dataset=='arc-challenge').values].mean().round(3),
        router_acc=acc, router_cost=cost, random_acc=racc, random_cost=rcost,
        acc_gain_vs_random=acc-racc, cost_saved_at_matching_acc=saved,
        pct_saved=100*saved/c_rand))

# bootstrap 95% CI for accuracy gain over random routing and for cost saved
rng = np.random.default_rng(42)
B = 2000
boot = {sh: {'gain': [], 'saved': []} for sh in shares}
for _ in range(B):
    idx = rng.integers(0, n, n)
    for sh in shares:
        acc, cost, _ = run(idx, sh)
        racc, _ = random_expect(idx, sh)
        sv, _ = saved_at_matching_acc(idx, acc, cost)
        boot[sh]['gain'].append(acc-racc); boot[sh]['saved'].append(sv)
for r in rows:
    g, s_ = boot[r['target_share']]['gain'], boot[r['target_share']]['saved']
    r['gain_ci_low'], r['gain_ci_high'] = np.percentile(g, [2.5, 97.5])
    r['saved_ci_low'], r['saved_ci_high'] = np.percentile(s_, [2.5, 97.5])

res = pd.DataFrame(rows)
res.to_csv(f'{folder}/results_router_vs_baselines.csv', index=False)

base = pd.DataFrame([
    dict(method='always_strong (GPT-4)', acc=S.mean(), cost=CS.sum()),
    dict(method='always_weak (Mixtral)', acc=W.mean(), cost=CW.sum()),
    dict(method='oracle (cheapest correct)', acc=np.maximum(S, W).mean(),
         cost=np.where(W >= S, CW, CS).sum()),
])
base.to_csv(f'{folder}/results_baselines.csv', index=False)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(base.round(4))
print(res[['target_share','cutoff_score','actual_share','share_gsm8k','share_arc',
           'router_acc','random_acc','acc_gain_vs_random','gain_ci_low','gain_ci_high']].round(4))
print(res[['target_share','router_cost','random_cost','cost_saved_at_matching_acc',
           'pct_saved','saved_ci_low','saved_ci_high']].round(4))

In [ ]:
import matplotlib.pyplot as plt

# Chart 1: accuracy vs cost
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(res.router_cost, res.router_acc, 'o-', label='BERT router (cut-offs 10-90%)')
ax.plot(res.random_cost, res.random_acc, 's--', color='gray', label='Random routing (expected)')
ax.scatter(base.cost[:2], base.acc[:2], c=['red','green'], s=90, zorder=5)
for _, r in base.iloc[:2].iterrows(): ax.annotate(r.method, (r.cost, r.acc), textcoords='offset points', xytext=(6,-12))
for _, r in res[res.target_share.isin([.1,.3,.5,.7])].iterrows():
    ax.annotate(f'{int(r.target_share*100)}%', (r.router_cost, r.router_acc), textcoords='offset points', xytext=(-4,6))
ax.set_xlabel('Total cost on 500 prompts (USD, RouterBench 2024 prices)'); ax.set_ylabel('Mean score')
ax.set_title('Accuracy vs cost'); ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/chart1_accuracy_vs_cost.png', dpi=200); plt.show()

# Chart 2: cost saved at matching accuracy (with 95% CI)
sel = res[res.target_share.isin([.1,.2,.3,.5,.7])]
fig, ax = plt.subplots(figsize=(7,5))
err = [sel.cost_saved_at_matching_acc - sel.saved_ci_low, sel.saved_ci_high - sel.cost_saved_at_matching_acc]
ax.bar([f'{int(s*100)}%' for s in sel.target_share], sel.cost_saved_at_matching_acc, yerr=err, capsize=4)
ax.axhline(0, color='k', lw=.8)
ax.set_xlabel('Share of prompts sent to GPT-4'); ax.set_ylabel('Cost saved vs random routing at same accuracy (USD)')
ax.set_title('Cost saved at matching accuracy (95% bootstrap CI)'); plt.tight_layout()
plt.savefig(f'{folder}/chart2_cost_saved.png', dpi=200); plt.show()

# Chart 3: strong-model share per dataset at each cut-off
fig, ax = plt.subplots(figsize=(7,5))
x = np.arange(len(sel)); w = 0.27
ax.bar(x-w, sel.actual_share, w, label='All prompts')
ax.bar(x, sel.share_gsm8k, w, label='GSM8K')
ax.bar(x+w, sel.share_arc, w, label='ARC-Challenge')
ax.set_xticks(x); ax.set_xticklabels([f'{int(s*100)}%' for s in sel.target_share])
ax.set_xlabel('Target share (cut-off set on these prompts)'); ax.set_ylabel('Share sent to GPT-4')
ax.set_title('Who gets sent to the strong model'); ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/chart3_strong_share.png', dpi=200); plt.show()

In [ ]:
for d in ['grade-school-math', 'arc-challenge']:
    sub = df[df.dataset == d].reset_index(drop=True)
    s, w, b = sub.score_strong.values, sub.score_weak.values, sub.bert_score.values
    order = np.argsort(-b)
    out = []
    for sh in [0.2, 0.5, 0.8]:
        k = int(round(sh * len(sub)))
        ts = np.zeros(len(sub), bool); ts[order[:k]] = True
        rt = np.where(ts, s, w).mean()
        rd = sh * s.mean() + (1 - sh) * w.mean()
        out.append(f'{int(sh*100)}%: router {rt:.3f} vs random {rd:.3f} (diff {rt-rd:+.3f})')
    print(d, len(sub), '|', ' | '.join(out))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q routellm huggingface_hub pandas scikit-learn
folder = '/content/drive/MyDrive/MLResearch'

In [ ]:
import pandas as pd, numpy as np, ast, os
chosen = pd.read_csv(f'{folder}/routerbench_clean.csv')

def extract_question(prompt, dataset):
    try:
        parts = ast.literal_eval(prompt)
        p = parts[-1] if isinstance(parts, (list, tuple)) else str(parts)
    except Exception:
        p = prompt.strip().strip('[]').strip('\'"').replace('\\n', '\n')
    if dataset == 'arc-challenge':
        p = p.split('Print only a single choice')[0]
    return p.strip()

chosen['question'] = [extract_question(p, d) for p, d in zip(chosen.prompt, chosen.dataset)]
print(chosen.shape, chosen.question.nunique())

In [ ]:
import os
try:
    bert
    print('using existing bert object')
except NameError:
    os.environ.setdefault('OPENAI_API_KEY', 'placeholder')
    from routellm.controller import Controller
    ctrl = Controller(routers=['bert'],
                      strong_model='gpt-4-1106-preview',
                      weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')
    bert = ctrl.routers['bert']
print(bert.calculate_strong_win_rate('What is 2+2?'))

In [ ]:
from tqdm import tqdm
part_path = f'{folder}/bert_scores_full_partial.csv'
done = pd.read_csv(part_path) if os.path.exists(part_path) else pd.DataFrame(columns=['row', 'bert_score'])
scores = dict(zip(done.row.astype(int), done.bert_score))
print('already scored:', len(scores))

def save_partial():
    pd.DataFrame({'row': list(scores.keys()), 'bert_score': list(scores.values())}).to_csv(part_path, index=False)

todo = [i for i in range(len(chosen)) if i not in scores]
for n, i in enumerate(tqdm(todo)):
    scores[i] = bert.calculate_strong_win_rate(chosen.question.iloc[i])
    if (n + 1) % 250 == 0:
        save_partial()
save_partial()
chosen['bert_score'] = [scores[i] for i in range(len(chosen))]
chosen.to_csv(f'{folder}/routerbench_scored_full.csv', index=False)
print(chosen.groupby('dataset').bert_score.describe())

In [ ]:
from tqdm import tqdm
N_GSM, N_ARC = 600, 400
sample = pd.concat([
    chosen[chosen.dataset == 'grade-school-math'].sample(N_GSM, random_state=42),
    chosen[chosen.dataset == 'arc-challenge'].sample(N_ARC, random_state=42),
]).sort_index()

# reuse earlier scores (keys are row numbers in `chosen`)
scores = {}
for p in [f'{folder}/bert_scores_sample_partial.csv', f'{folder}/bert_scores_1000_partial.csv']:
    if os.path.exists(p):
        d_ = pd.read_csv(p)
        scores.update(dict(zip(d_.row.astype(int), d_.bert_score)))
print('already scored:', len([i for i in sample.index if i in scores]), 'of', len(sample))

def save_partial():
    pd.DataFrame({'row': list(scores.keys()), 'bert_score': list(scores.values())}).to_csv(
        f'{folder}/bert_scores_1000_partial.csv', index=False)

todo = [i for i in sample.index if i not in scores]
for n, i in enumerate(tqdm(todo)):
    scores[i] = bert.calculate_strong_win_rate(chosen.question.loc[i])
    if (n + 1) % 100 == 0:
        save_partial()
save_partial()

sample['bert_score'] = [scores[i] for i in sample.index]
sample.to_csv(f'{folder}/routerbench_scored_1000.csv', index=False)
print(len(sample), sample.dataset.value_counts().to_dict())
print(sample.groupby('dataset').bert_score.describe())

In [ ]:
df = pd.read_csv(f'{folder}/routerbench_scored_1000.csv').reset_index(drop=True)
S, W = df.score_strong.values, df.score_weak.values
CS, CW = df.cost_strong.values, df.cost_weak.values
sc, ds = df.bert_score.values, df.dataset.values
SHARES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
B = 2000
GSM, ARC = 'grade-school-math', 'arc-challenge'

def evaluate(s, w, cs, cw, b, shares):
    """Send the top `share` of prompts (by BERT score) to the strong model."""
    n = len(s); order = np.argsort(-b, kind='stable')
    a_s, a_w = s.mean(), w.mean()
    out = []
    for sh in shares:
        k = int(round(sh * n))
        ts = np.zeros(n, bool); ts[order[:k]] = True
        acc = np.where(ts, s, w).mean(); cost = np.where(ts, cs, cw).sum()
        racc = sh * a_s + (1 - sh) * a_w
        rcost = sh * cs.sum() + (1 - sh) * cw.sum()
        q = np.clip((acc - a_w) / (a_s - a_w), 0, 1) if a_s != a_w else 0.0
        crand = q * cs.sum() + (1 - q) * cw.sum()   # random mix that matches router accuracy
        out.append((acc, cost, racc, rcost, crand - cost, crand, ts.mean()))
    return np.array(out)

def bootstrap(mask, shares, B, seed=42):
    idx_all = np.where(mask)[0]; rng = np.random.default_rng(seed)
    g, sv = [], []
    for _ in range(B):
        idx = rng.choice(idx_all, len(idx_all))
        r = evaluate(S[idx], W[idx], CS[idx], CW[idx], sc[idx], shares)
        g.append(r[:, 0] - r[:, 2]); sv.append(r[:, 4])
    g, sv = np.array(g), np.array(sv)
    return np.percentile(g, [2.5, 97.5], axis=0), np.percentile(sv, [2.5, 97.5], axis=0)

print(len(df), df.dataset.value_counts().to_dict())

In [ ]:
base = pd.DataFrame([
    dict(method='always_strong (GPT-4)', acc=S.mean(), cost=CS.sum()),
    dict(method='always_weak (Mixtral)', acc=W.mean(), cost=CW.sum()),
    dict(method='oracle (cheapest correct)', acc=np.maximum(S, W).mean(), cost=np.where(W >= S, CW, CS).sum()),
])
base.to_csv(f'{folder}/full_results_baselines.csv', index=False)

r = evaluate(S, W, CS, CW, sc, SHARES)
g_ci, s_ci = bootstrap(np.ones(len(df), bool), SHARES, B)

order_all = np.argsort(-sc, kind='stable')
cut, sh_gsm, sh_arc = [], [], []
for sh in SHARES:
    k = int(round(sh * len(sc)))
    top = np.zeros(len(sc), bool); top[order_all[:k]] = True
    cut.append(sc[order_all[k - 1]])
    sh_gsm.append(top[ds == GSM].mean()); sh_arc.append(top[ds == ARC].mean())

res = pd.DataFrame(dict(
    target_share=SHARES, cutoff_score=cut, actual_share=r[:, 6],
    share_gsm8k=sh_gsm, share_arc=sh_arc,
    router_acc=r[:, 0], random_acc=r[:, 2], acc_gain_vs_random=r[:, 0] - r[:, 2],
    gain_ci_low=g_ci[0], gain_ci_high=g_ci[1],
    router_cost=r[:, 1], random_cost=r[:, 3],
    cost_saved_at_matching_acc=r[:, 4], pct_saved=100 * r[:, 4] / r[:, 5],
    saved_ci_low=s_ci[0], saved_ci_high=s_ci[1]))
res.to_csv(f'{folder}/full_results_router_vs_baselines.csv', index=False)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(base.round(4))
print(res[['target_share','cutoff_score','actual_share','share_gsm8k','share_arc','router_acc','random_acc','acc_gain_vs_random','gain_ci_low','gain_ci_high']].round(4))
print(res[['target_share','router_cost','random_cost','cost_saved_at_matching_acc','pct_saved','saved_ci_low','saved_ci_high']].round(4))
print('Accuracy-gain CIs that include 0:', int(((res.gain_ci_low <= 0) & (res.gain_ci_high >= 0)).sum()), 'of', len(res))
print('Cost-saved CIs that exclude 0:', int(((res.saved_ci_low > 0) | (res.saved_ci_high < 0)).sum()), 'of', len(res))

In [ ]:
import time
print(len(df))
t = time.time()
for _ in range(200):
    evaluate(S, W, CS, CW, sc, SHARES)
print('seconds for 200 calls:', round(time.time() - t, 2))

In [ ]:
rows = []
for d in [GSM, ARC]:
    m = ds == d; idx = np.where(m)[0]
    r = evaluate(S[idx], W[idx], CS[idx], CW[idx], sc[idx], SHARES)
    g_ci, s_ci = bootstrap(m, SHARES, B)
    for j, sh in enumerate(SHARES):
        rows.append(dict(dataset=d, n=len(idx), target_share=sh,
                         router_acc=r[j, 0], random_acc=r[j, 2], gain=r[j, 0] - r[j, 2],
                         gain_ci_low=g_ci[0, j], gain_ci_high=g_ci[1, j],
                         cost_saved=r[j, 4], saved_ci_low=s_ci[0, j], saved_ci_high=s_ci[1, j]))
perds = pd.DataFrame(rows)
perds.to_csv(f'{folder}/full_results_per_dataset.csv', index=False)
print(perds.round(4).to_string(index=False))

In [ ]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(42)

def auc_ci(y, x, reps=1000):
    if len(set(y)) < 2: return np.nan, np.nan, np.nan
    a = roc_auc_score(y, x); bs = []
    for _ in range(reps):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: bs.append(roc_auc_score(y[i], x[i]))
    return a, *np.percentile(bs, [2.5, 97.5])

out = []
for name, m in [('pooled', np.ones(len(df), bool)), ('GSM8K only', ds == GSM), ('ARC only', ds == ARC)]:
    y = (S[m] > W[m]).astype(int)   # 1 = GPT-4 strictly better than Mixtral on this prompt
    a, lo, hi = auc_ci(y, sc[m])
    out.append(dict(subset=name, n=int(m.sum()), share_strong_better=round(y.mean(), 3),
                    auc=round(a, 3), ci_low=round(lo, 3), ci_high=round(hi, 3)))
a, lo, hi = auc_ci((ds == GSM).astype(int), sc)
out.append(dict(subset='score separates GSM8K from ARC', n=len(df), share_strong_better=np.nan,
                auc=round(a, 3), ci_low=round(lo, 3), ci_high=round(hi, 3)))
aucs = pd.DataFrame(out); aucs.to_csv(f'{folder}/full_results_auc.csv', index=False)
print(aucs.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Chart 1: accuracy vs cost
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(res.router_cost, res.router_acc, 'o-', label='BERT router (cut-offs 10-90%)')
ax.plot(res.random_cost, res.random_acc, 's--', color='gray', label='Random routing (expected)')
ax.scatter(base.cost[:2], base.acc[:2], c=['red', 'green'], s=90, zorder=5)
for _, r_ in base.iloc[:2].iterrows():
    ax.annotate(r_.method, (r_.cost, r_.acc), textcoords='offset points', xytext=(6, -12))
for _, r_ in res[res.target_share.isin([.1, .3, .5, .7])].iterrows():
    ax.annotate(f'{int(r_.target_share*100)}%', (r_.router_cost, r_.router_acc), textcoords='offset points', xytext=(-4, 6))
ax.set_xlabel(f'Total cost on {len(df)} prompts (USD, RouterBench 2024 prices)'); ax.set_ylabel('Mean score')
ax.set_title('Accuracy vs cost'); ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/full_chart1_accuracy_vs_cost.png', dpi=200); plt.show()

# Chart 2: cost saved at matching accuracy (negative = router costs more)
sel = res[res.target_share.isin([.1, .2, .3, .5, .7, .9])]
err = [sel.cost_saved_at_matching_acc - sel.saved_ci_low, sel.saved_ci_high - sel.cost_saved_at_matching_acc]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.bar([f'{int(s*100)}%' for s in sel.target_share], sel.cost_saved_at_matching_acc, yerr=err, capsize=4)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Target share of prompts sent to GPT-4')
ax.set_ylabel('Cost saved vs random (USD)\nnegative = router costs more')
ax.set_title('Cost saved at matching accuracy (95% CI)'); plt.tight_layout()
plt.savefig(f'{folder}/full_chart2_cost_saved.png', dpi=200); plt.show()

# Chart 3: who gets sent to the strong model
fig, ax = plt.subplots(figsize=(7.5, 5))
x = np.arange(len(res)); wd = 0.38
ax.bar(x - wd/2, res.share_gsm8k, wd, label='GSM8K')
ax.bar(x + wd/2, res.share_arc, wd, label='ARC-Challenge')
ax.set_xticks(x); ax.set_xticklabels([f'{int(s*100)}%' for s in res.target_share])
ax.set_xlabel('Target share of prompts sent to GPT-4'); ax.set_ylabel('Share of that dataset sent to GPT-4')
ax.set_title('Who gets sent to the strong model'); ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/full_chart3_who_gets_strong.png', dpi=200); plt.show()

# Chart 4: accuracy gain over random, per dataset
fig, ax = plt.subplots(figsize=(7.5, 5))
for d, lab in [(GSM, 'GSM8K'), (ARC, 'ARC-Challenge')]:
    p = perds[perds.dataset == d]
    ax.errorbar(p.target_share * 100, p.gain, yerr=[p.gain - p.gain_ci_low, p.gain_ci_high - p.gain],
                marker='o', capsize=3, label=lab)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Target share sent to GPT-4 (%)'); ax.set_ylabel('Accuracy gain over random (95% CI)')
ax.set_title('Within-dataset accuracy gain'); ax.legend(); plt.tight_layout()
plt.savefig(f'{folder}/full_chart4_per_dataset_gain.png', dpi=200); plt.show()

In [ ]:
from getpass import getpass
from openai import OpenAI

KEY = getpass('Paste your NEW key and press Enter: ')
client = OpenAI(
    api_key=KEY,
    base_url='https://YOUR-RESOURCE.openai.azure.com/openai/v1/'
)
e = client.embeddings.create(input=['What is 2+2?'], model='text-embedding-3-small')
print(len(e.data[0].embedding))   # should print 1536

In [ ]:
for name in ['sc', 'df', 'folder', 'evaluate', 'client']:
    print(name, 'OK' if name in globals() else 'MISSING')
try:
    import routellm; print('routellm OK')
except ImportError:
    print('routellm MISSING')

In [ ]:
import numpy as np
from tqdm import tqdm

# router_scores may not exist yet if you skipped Cells 10-11
try:
    router_scores
except NameError:
    router_scores = {'BERT (RouteLLM, pretrained)': sc}

# point RouteLLM's three modules at your Foundry client
import routellm.routers.similarity_weighted.utils as u1
import routellm.routers.routers as u2
import routellm.routers.matrix_factorization.model as u3
for m in (u1, u2, u3):
    m.OPENAI_CLIENT = client

from routellm.controller import Controller
ctrl2 = Controller(routers=['mf', 'sw_ranking'],
                   strong_model='gpt-4-1106-preview',
                   weak_model='mistralai/Mixtral-8x7B-Instruct-v0.1')

mf = ctrl2.routers['mf']
vals = [mf.calculate_strong_win_rate(q) for q in tqdm(df.question, desc='mf')]
df['mf_score'] = vals
router_scores['RouteLLM mf'] = np.array(vals)
df.to_csv(f'{folder}/routerbench_scored_1000_all_routers.csv', index=False)
print(df[['bert_score', 'mf_score']].describe())

In [ ]:
import time
sw = ctrl2.routers['sw_ranking']
t = time.time()
test_vals = [sw.calculate_strong_win_rate(q) for q in df.question.iloc[:10]]
dt = (time.time() - t) / 10
print('scores:', np.round(test_vals, 3))
print(f'{dt:.2f} s per prompt -> about {dt*1000/60:.0f} minutes for 1,000 prompts')

In [ ]:
!pip install "routellm[serve,eval]" -q

In [ ]:
!pip install routellm -q

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from routellm.controller import Controller
print('routellm OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, os
from tqdm import tqdm

folder = '/content/drive/MyDrive/MLResearch'   # use your real folder path
df = pd.read_csv(f'{folder}/routerbench_scored_1000_all_routers.csv')
sc = df['bert_score'].values
router_scores = {'BERT (RouteLLM, pretrained)': sc}
if 'mf_score' in df.columns:
    router_scores['RouteLLM mf'] = df['mf_score'].values
print(df.shape, [c for c in df.columns if 'score' in c])

In [ ]:
import os, time
from concurrent.futures import ThreadPoolExecutor
print('CPU cores:', os.cpu_count())

qs = list(df.question.iloc[10:30])
t = time.time()
with ThreadPoolExecutor(4) as ex:
    out = list(ex.map(sw.calculate_strong_win_rate, qs))
print(f'{(time.time()-t)/20:.2f} s per prompt with 4 threads (was 5.29)')

In [ ]:
for name in ['df', 'sw', 'ctrl2', 'client', 'sc', 'folder', 'evaluate']:
    print(name, 'OK' if name in globals() else 'MISSING')

In [ ]:
import os, pandas as pd
sw_part = f'{folder}/sw_ranking_partial.csv'
done = pd.read_csv(sw_part) if os.path.exists(sw_part) else pd.DataFrame(columns=['row', 'score'])
sw_scores = dict(zip(done.row.astype(int), done.score))
print('already scored:', len(sw_scores))

def save_sw():
    pd.DataFrame({'row': list(sw_scores.keys()), 'score': list(sw_scores.values())}).to_csv(sw_part, index=False)

todo = [i for i in range(len(df)) if i not in sw_scores]
for n, i in enumerate(tqdm(todo, desc='sw_ranking')):
    sw_scores[i] = sw.calculate_strong_win_rate(df.question.iloc[i])
    if (n + 1) % 25 == 0:
        save_sw()
save_sw()
df['sw_ranking_score'] = [sw_scores[i] for i in range(len(df))]
router_scores['RouteLLM sw_ranking'] = df['sw_ranking_score'].values
df.to_csv(f'{folder}/routerbench_scored_1000_all_routers.csv', index=False)
print(df[['bert_score', 'mf_score', 'sw_ranking_score']].describe())